#Корпус решений Верховного суда Нидерландов по гражданским делам за 2025 год

Корпус решений Верховного суда Нидерландов (Hoge Raad der Nederlanden) нужен для выполнения чат-ботом своей второй функции - выдачи по запросу пользователя текста судебного решения определенного размера и уровня лексического разнообразия со списком ключевых слов.

#1. Загрузка библиотек

In [1]:
!pip install stanza
import stanza
stanza.download('nl')
nlp = stanza.Pipeline(lang = 'nl')

!pip install pandas
import pandas as pd

from collections import Counter

import os

import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.6 MB/s eta 0:00:00


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: nl (Dutch) ...


INFO:stanza:Downloaded file to /root/stanza_resources/nl/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: nl (Dutch):
| Processor | Package         |
-------------------------------
| tokenize  | alpino          |
| mwt       | alpino          |
| pos       | alpino_charlm   |
| lemma     | alpino_nocharlm |
| depparse  | alpino_charlm   |
| ner       | conll02         |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
stop_words = stopwords.words('dutch')
print(stop_words[:10])

['de', 'en', 'van', 'ik', 'te', 'dat', 'die', 'in', 'een', 'hij']


In [5]:
from collections import Counter

#2. Определение функций

*Пример работы функций приведен в разделе 5.*

##2.1. Первичная обработка текста - initial_clean_the_text
Функция разделяет "слипшиеся" буквы и цифры и корректирует отображение в тексте нумерации.


Она будет использоваться для чистки текста перед выдачей его пользователю, а также для первичной обработки текста перед направлением его в Stanza и подсчетом TTR и TF-IDF.

In [6]:
def initial_clean_the_text(doc):
  doc_clean = re.sub(r'(?<=\d)(?=[A-Z])', '. ', doc)
  doc_clean = re.sub(r'(?<=[A-Za-z])(?=\d+)', ' ', doc_clean)
  doc_clean = re.sub(r'(?<=\d\.\d)\s', '. ', doc_clean)
  doc_clean = re.sub(r'(?<=[A-Za-z]\.)(?=\d+)', ' ', doc_clean)

  return doc_clean

##2.2. Финальная обработка текста - final_clean_the_text
Функция применяется для обработки текста перед направлением его в Stanza и подсчетом TTR и TF-IDF. В ней важно было соблюсти баланс между чисткой текста и сохранением необходимого материала для корректной работы Stanza.

Функция выполняет четыре основных задачи:  
(1) заменяет обезличенные фрагменты текстов судебных решений на заглушку-NER, чтобы в дальшейшем их удалить, как это будет сделано с NER в других текстах;  
(2) убирает лишний шум (коды судебных решений, нумерацию);  
(3) вносит в текст небольшие правки, чтобы Stanza корректно проставила частеречную и синтаксическую разметку;  
(4) обрезает текст судебного решения до слов "судебное решение вынесено на открытом заседании судьей..." (после этих слов следуют сноски, которые неважны для целей создания корпуса).

Для решения задачи (1) дополнительно были собраны данные относительно того, какие фрагменты текстов судебных решений суд обычно помещает в квадратные скобки - только обезличенные данные об участниках процесса или же ещё какие-либо элементы текста. Выяснилось, что в квадратные скобки суд также помещает ошибки, опечатки, названия законов, указания на объекты недвижимости. В связи с этим чистка текста от обезличенных данных была проведена адресно, только в отношении тех фрагментов, которые можно отнести к NER (подробнее см. раздел 3 тетрадки).

Задача (3) появилась после того, как в работе Stanza были обнаружены неточности, которые, однако, можно устранить. Например:  
- Stanza хорошо распознает NER, когда инициала всего 2, когда же их 3-4, она начинает неверно проставлять разметку, поэтому там, где в имени человека 3-4 инициала, его имя заменяется на заглушку-NER;  
- слова, написанные полностью с заглавных букв, Stanza неверно лемматизирует, вплоть до того, что в качестве леммы указываются несуществующие слова, в связи с этим названия участников процесса (истец, ответчик и пр.) были приведены полностью к нижнему регистру, а в остальных словах, состоящих только из заглавных букв, заглавной была оставлена только первая буква, чтобы Stanza могла распознать их как потенциальный NER.


In [7]:
def final_clean_the_text(doc_clean):
  final_doc_clean = re.sub(r'ECLI:NL:\w+:\d+:\d+', ' ', doc_clean)
  final_doc_clean = re.sub(r'ECLI:NL:\w+:\d+', ' ', final_doc_clean)

  final_doc_clean = re.sub(r'(?<=\sin de zaak\s)[\d\\\sA-Zenzak\./\(\)-]+?(?=\svan het)', 'nummer 123', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=\sin de zaken\s)[\d\\\sA-Zenzak\./\(\)-]+?(?=\svan het)', 'nummer 123', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=\sin de zaak\s)[\d\\\sA-Zenzak\./\(\)-]+?(?=\svan de)', 'nummer 123', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=\sin de zaken\s)[\d\\\sA-Zenzak\./\(\)-]+?(?=\svan de)', 'nummer 123', final_doc_clean)

  patterns = [r'(?<=\nadvocaat: )[\w\s\.,&-]+?(?=[\.,]\n)', r'(?<=is gegeven door )[A-Za-z\s\.,-]+?(?= en in het openbaar uitgesproken)', r'(?<=Advocaat-Generaal\s)[A-Za-z\.\s-]+?(?=\sstrekt)', r'(?<=\n\n\d\.\s)[A-Z][\w\s\.&-]+?(?=[\.,]\n\nwonende)', r'(?<=\nIn de zaak van\n\n)[A-Z][\w\s\.&-]+?(?=[\.,]\n)', r'(?<=\nhierna:\s)[A-Z][\w\s\.&-]+?(?=[\.,]\n)', r'(?<=\ntegen\n\n)[A-Z][\w\s\.&-]+?(?=[\.,]\n)', r'(?<=\nIn de zaak van\n\n\d\.\s)[A-Z][\w\s\.&-]+?(?=[\.,]\n)', r'(?<=\n\n\d\.\s)[A-Z][\[\]\w\s\.&-]+?(?=[\.,]\n\ngevestigd)', r'(?<=\n\n\d\d\.\s)[A-Z][\[\]\w\s\.&-]+?(?=[\.,]\n\ngevestigd)']
  c = 0
  sub_names = ['Tom', 'Anneke', 'Marijke', 'Gert', 'Guus', 'Willem', 'Hanna', 'Pieter']

  for i in patterns:
    res = re.findall(i, final_doc_clean)
    if len(res) > 0:
      for name in res:
        if c == 8:
          c = 0
        final_doc_clean = re.sub(fr'{name}', f' {sub_names[c]} ', final_doc_clean)
        c = c + 1

  patterns_plural = [r'(?<=\nadvocaten: )[\w\s\.-]+?(?=[\.,]\n)', r'(?<=\n\nhierna gezamenlijk:\s)[A-Z][\w\s\.&-]+?(?=[\.,]\n\n)']
  c = 0
  sub_names_plural = ['Tom en Anneke', 'Anneke en Marijke', 'Marijke en Gert', 'Gert en Guus', 'Guus en Willem', 'Willem en Hanna', 'Hanna en Pieter', 'Pieter en Tom']

  for i in patterns_plural:
    res = re.findall(i, final_doc_clean)
    if len(res) > 0:
      for name in res:
        if c == 8:
          c = 0
        final_doc_clean = re.sub(fr'{name}', f' {sub_names_plural[c]} ', final_doc_clean)
        c = c + 1

  list_afkortingen = ['AAW','ABW','AKW','AMvB','Anw','AOW','APK','ARAR','Ai','Arbowet','ATW','Aw','Awb','Avv','Avv','AWBZ','AWIR','AWR','AWW','BAG','BAPO','BBSH','Bevi','BHW','BIBOB','BIG', 'BKH','BKR','Bopz','BPM','BRI','BRV','BS','BW','BZM','CAO','CFI','DivB','DW','FinBES','FLO','FPU','Fw','GAK','GBLT','GemW','GrW','HPW','HUURT','Hvb','Hvw', 'Hw','IB 2001','IBES','Int.','Int. bijst. verl.', 'Inv','KSB','K.B.','KO','LB','Lw','LW69','MilH','Minb','MRB','OB','ozb','Opw','Ow','ProvW','Pw', 'RDH', 'RO','Rv','RWN','StuFi','SV','SW','tbs','Toesl','TOG','TSZ','UHW','VUT','Vpb 1969','W.A. of WA','WABM', 'Waadi', 'Wabo','Wajong','Walvis','WAM','WAO','WAT', 'Waz','Wazo','Wbbbg','Wbg','WBO','Wbp','WBSO','WEB','Wet IB 2001','Wet MOT','Wet WOZ','Wft','WGR','Wgr','WHC','WHV','WHW','WIA','WION','Wiv 2002','Wiv 2017', 'WKA','WKCZ','Wlz','WM','WMO','Wmo 2015','Wnt','WOB','WOHV','WolBES','WOO','WOR','Wp2000','WRO','Wro','Wsnp','Wta','WTU-BES','Wvggz','WVGS', 'WvK','WVMK','WvS','WVW','Wonw','WW','WWB','Wwft','Wwik','WWM','WWOZ','WWZ','Wzd','ZFW','ZORGV','ZW','Zvw']

  for i in list_afkortingen:
    final_doc_clean = re.sub(fr'(?<=\b){i}(?=\b)', ' wetboek ' , final_doc_clean)

  final_doc_clean = re.sub(r'([A-Z]\.)?[A-Z]\.[A-Z]\.[A-Z]\.\s(\w+\s)?[A-Z][a-z]+', ' Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\b(\d)?\d\.', ' ', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=[a-z])\(en\)', 'en', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=[a-z])\(s\)', 's', final_doc_clean)
  final_doc_clean = re.sub(r'(?<=[a-z])/(?=[a-z])', ' of ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[A-Z]\]|\[[A-Z][A-Z]\]', ' Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[A-Z][A-Z]\s[a-z]\.[a-z]\.\]', ' Marijke en Anneke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[lL]eeftijd\]', ' 5 ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[mM]aand, jaar\]', ' april 1931 ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[eE]iser\]|\[[eE]iseres\]|\[[eE]iser \([\w\s/]+\)\]|\[[eE]iser \d+\]|\[[eE]iseres \d+\]|\[[vV]erzoek(er|ster) \d+\]|\[[vV]erzoek(er|ster)\]|\[[Aa]ppellant\]|\[[Cc]hauffeur\]', ' Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[eE]isers\]|\[[eE]iseressen\]|\[[eE]isers \([\w\s/]+\)\]|\[[vV]erzoek(ers|sters)([\s\w/]+)?\]', ' Marijke en Anneke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[vV]erweer(der|ster)\]|\[[vV]erweer(der|ster) \([\w\s/]+\)\]|\[[vV]erweer(der|ster) \d+([a-z])?\]|\[[bB]elanghebbende\]|\[[bB]elanghebbende \d+\]|\[[Bb]etrokkene\]|\[[Bb]etrokkene \d+\]', ' Anneke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[vV]erweer(der|ster)s([\s\w/]+)?\]|\[[vV]erweer(der|ster)s \([\w\s/]+\)\]|\[[bB]elanghebbenden\]|\[[bB]etrokkenen ([\s\w/]+)?\]', ' Willem en Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([Dd]e )?[Ww]erkne(emster|mer)\]|\[[Bb]roer( \d+)?\]|/[een derde/]', ' Tom ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([dD]e )?(zus|dochter|zoon|broer|zakenpartner|halfbroer|moeder|statutair bestuurder|bewindvoerder|koper|benadeeelde|oma|buurman|zorgaanbieder|minderjarige|schoonmaakster|vereffenaar|ingeleende|leidinggevende|agente|juridische dienstverlener|bewoner|verhuurder|man|Fysiotherapeut|vrouw|echtgenote|vader|derde|advocaat|commissaris|bestuurder|familie)( [\dA-Z]+)?\]', ' Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([dD]e |[bB]edoelde |[Bb]epaalde |[Dd]eze )?(ouders|eisers)\]', ' Anneke en Guus ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([dD]e zus en de broer)|([dD]e broer en de zus)|([dD]e vader en de moeder)|([dD]e moeder en de vader)|([kK]inderen van overleden huurster)|([dD]e erven)\]', ' Anneke en Marijke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[lL]and\]', ' Nederland ', final_doc_clean)
  final_doc_clean = re.sub(r'h\.o\.d\.n(\.)?', ' ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([Bb]eheer|[Bb]edrijf|[vV]ennootschap|de voormalige vennoot|v\.o\.f(\.)?)( \d+)?\]', ' Tom ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[Nn]aam( klant)?\]', ' Anneke ', final_doc_clean)
  final_doc_clean = re.sub(r'\[([vV]estigings|[wW]oon|[Vv]erblijf|[Gg]eboorte)plaats\]', ' Amsterdam ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[pP]laats(naam)?\]', ' Rotterdam ', final_doc_clean)
  final_doc_clean = re.sub(r'\[[ixv]+\]', ' ', final_doc_clean)
  final_doc_clean = re.sub(r'\]|\[', ' ', final_doc_clean)
  final_doc_clean = re.sub(r'\([ixv]+\)', ' ', final_doc_clean)
  final_doc_clean = final_doc_clean.replace('Nummer', 'nummer').replace('Datum', 'datum').replace('EISERESSEN', 'eiseressen').replace('EISERES', 'eiseres').replace('EISERS', 'eisers').replace('EISER', 'eiser').replace('VERWEERSTERS', 'verweersters').replace('VERWEERDERS', 'verweerders').replace('VERWEERSTER', 'verweerster').replace('VERWEERDER', 'verweerder')

  naive_tokens = []
  for i in final_doc_clean.split():
      if i.isupper() and '.' not in i:
        naive_tokens.append(f'{i[0]}{i[1:].lower()}')
      else:
        naive_tokens.append(i)

  final_doc_clean = ' '.join(naive_tokens)

  final_doc_clean = re.sub(r'in het openbaar uitgesproken door de [\s\S]*', 'in het openbaar uitgesproken door de ', final_doc_clean)

  return final_doc_clean

##2.3. Обработка текста в Stanza и получение первичного дата-фрейма - get_data_frame_common
Функция собирает материал для дальнейшего подсчета TTR текста судебного решения и первичный материал для подсчета TF-IDF.

Пошагово внутри функции происходит следующее:  
(1) текст обрабатывается в Stanza;  
(2) функция итерируется по результатам работы Stanza: отдельно по частеречной разметке (в sentence.words), отдельно по NER (в sentence.tokens), и формирует 2 дата-фрейма, которые затем соединяются в один по sentenсe id и word id; sentenсe id формируется с помощью count, поскольку в Stanza нет нумерации предложений;  
(3) дата-фрейм несколько раз фильтруется, чтобы убрать из него:  
- служебные части речи, NER, символы, числа;
- стоп-слова (взяты из библиотеки NLTK, дополнительного списка стоп-слов, найденного в интернете - [здесь](https://github.com/stopwords-iso/stopwords-nl), также добавила отдельные стоп-слова из текстов судебных решений, которые выявила при анализе текстов);  
- лишний шум (слова менее 3 символов, которые обычно представляют собой сокращения и нераспознанные Stanza части NER).


In [8]:
def get_data_frame_common(cleaned_document):
  stanza_clean = nlp(cleaned_document)
  for_data_frame_pos = []
  count = 0
  for sentence in stanza_clean.sentences:
      for word in sentence.words:
          for_data_frame_pos.append({'sentence_id': count, 'word_id': word.id, 'word': word.text,
                                'lemma': word.lemma, 'upos': word.upos, 'xpos': word.xpos,
                                'head': word.head, 'deprel': word.deprel})
      count = count + 1
  df_pos = pd.DataFrame(for_data_frame_pos)

  for_data_frame_ner = []
  count = 0
  for sentence in stanza_clean.sentences:
      for token in sentence.tokens:
          for_data_frame_ner.append({'sentence_id': count, 'word_id': token.id[0], 'word': token.text, 'ner': token.ner})
      count = count + 1
  df_ner = pd.DataFrame(for_data_frame_ner)
  df_ner_selected = df_ner[['sentence_id', 'word_id', 'ner']]

  merged_df_pos_ner = pd.merge(df_pos, df_ner_selected, on=['sentence_id', 'word_id'], how='outer')

  del_pos = ['NUM', 'X', 'SYM', 'PUNCT', 'DET', 'AUX', 'PRON', 'CCONJ', 'INTJ', 'ADP', 'SCONJ']
  merged_df_pos_ner_filtered = merged_df_pos_ner.loc[~merged_df_pos_ner['upos'].isin(del_pos)]

  del_ner = ['B-PER', 'I-PER', 'E-PER', 'S-PER', 'B-ORG', 'I-ORG', 'E-ORG', 'S-ORG', 'B-LOC', 'I-LOC', 'E-LOC', 'S-LOC', 'I-MISC', 'B-MISC', 'E-MISC', 'S-MISC']
  merged_df_pos_ner_filtered_2 = merged_df_pos_ner_filtered.loc[~merged_df_pos_ner_filtered['ner'].isin(del_ner)].copy()

  merged_df_pos_ner_filtered_3 = merged_df_pos_ner_filtered_2.loc[merged_df_pos_ner_filtered_2['lemma'].str.len() > 2]

  merged_df_pos_ner_filtered_4 = merged_df_pos_ner_filtered_3[~merged_df_pos_ner_filtered_3['word'].str.contains(r'\d+', regex=True)].copy()

  merged_df_pos_ner_filtered_4['lemma'] = merged_df_pos_ner_filtered_4['lemma'].str.lower()

  merged_df_pos_ner_filtered_4['word'] = merged_df_pos_ner_filtered_4['word'].str.lower()

  # в стоп-словах здесь и далее могут попадаться артефакты из первых чисток текста, когда я что-то удаляла вручную и
  # лишь потом заменила на регулярку/функцию, которая такое удаление делает за меня
  # я постаралась их убрать, но могла что-то недоглядеть, поэтому там могут попадаться странные буквосочетания
  del_word = stop_words + ['zult', 'betreffende', 'evenmin', 'vallen', 'vier', 'zeer', 'aldaar', 'miss', 'aangezien', 'voort', 'prof.', 'drs.', 'indien', 'rato', 're', 'dat', 'duizend', 'vanwege', 'tom', 'niets', 'hiervan', 'al', 'blijkbaar', 'altijd', 'jij', 'urw', 'terzijde', 'vooruit', 'omlaag', 'meestal', 'jouwe', 'wederom', 'meest', 'zowat', 'doen', 'zijn', 'ik', 'neem', 'tenzij', 'nee', 'hierbeneden', 'daarvoor', 'hierom', 'niks', 'ons', 'wanneer', 'mocht', 'tien', 'even', 'zulk', 'dertig', 'tot', 'voorbij', 'sbk', 'nadat', 'gekund', 'overal', 'ov', 'ehg-nl', 'niemand', 'alias', 'zelfs', 'gedurende', 'kan', 'zei', 'jouw', 'kunnen', 'samen', 'hoe', 'rond', 'mijn', 'werden', 'toch', 'hare', 'zouden', 'meer', 'noch', 'etc', 'was', 'zal', 'wijzelf', 'achte', 'anderszins', 'beiden', 'niet', 'lijkt', 'sinds', 'wzd', 'weinig', 'mrs', 'is', 'er', 'waar', 'waarboven', 'doorgaand', 'mr.', 'ms', 'ms.', 'pmt', 'lwb', 'rzv', 'dr', 'dr.', 'bns', 'bwa', 'der', 'nederlanden', 'aua', 'wzd', 'clni', 'llmc','timp', 'nl.', 'bns', 'kvz', 'rome', 'cur', 'wzd', 'wvggz', 'sbk', 'dws', 'nogal', 'wel', 'beide', 'llmc', 'zijzelf', 'anders', 'daarvanlangs', 'de', 'uw', 'nu', 'daar', 'missen', 'aan', 'maken', 'waarbij', 'ander', 'nergens', 'honderd', 'krachtens', 'waarvan', 'toe', 'eerlang', 'lifex', 'aangaande', 'na', 'voorts', 'wat', 'word', 'iedereen', 'daarnet', 'inmiddels', 'je', 'hoewel', 'heb', 'ja', 'af', 'misschien', 'nemen', 'en', 'lgr', 'wie', 'enig', 'moet', 'maar', 'verscheidene', 'iets', 'alles', 'kun', 'nooit', 'zodat', 'bovenvermeld', 'daarheen', 'aldus', 'jullie', 'doe', 'zonder', 'kunt', 'hebben', 'hunne', 'vooralsnog', 'bijna', 'ooit', 'bv', 'zichzelf', 'desbetreffende', 'zeven', 'tijdens', 'waarschijnlijk', 'eer', 'eerst', 'slechts', 'omdat', 'jegens', 'daarna', 'laatst', 'andere', 'kon', 'bijv', 'echter', 'wil', 'bns', 'erdoor', 'deden', 'gewoon', 'eigenlijk', 'als', 'liever', 'derde', 'onder', 'steeds', 'overige', 'vol', 'daarom', 'drie', 'eens', 'het', 'zelf', 'wegens', 'tweede', 'vooraf', 'zulks', 'klaar', 'juist', 'zojuist', 'inderdaad', 'geleden', 'per', 'mogen', 'enigszins', 'deed', 'waarover', 'rondom', 'gewoonweg', 'negen', 'gauw', 'vijftig', 'ongeveer', 'heel', 'konden', 'vanuit', 'boven', 'haarzelf', 'maakt', 'want', 'zijne', 'hun', 'behalve', 'gelijk', 'moest', 'door', 'verschillende', 'ondertussen', 'alleen', 'totdat', 'elk', 'voor', 'nv', 'werd', 'onzeker', 'toenmaals', 'hierna', 'om', 'zoals', 'jou', 'te', 'zullen', 'willen', 'maakte', 'waren', 'recent', 'weldra', 'nochtans', 'binnenin', 'ge', 'voorop', 'beneden', 'dus', 'elke', 'vandaan', 'we', 'immers', 'die', 'wordt', 'waarnaar', 'ieder', 'hij', 'verder', 'dan', 'bent', 'toenmalig', 'tegenover', 'waarvoor', 'vaakwat', 'bepaald', 'ergens', 'alle', 'eerder', 'overeind', 'zeker', 'clni', 'enige', 'erg', 'pmt', 'hen', 'waarom', 'overigens', 'vanaf', 'onderhavige', 'voorheen', 'daardoor', 'vrij', 'redelijk', 'vervolgens', 'bovengenoemd', 'alhoewel', 'zij', 'achter', 'iedere', 'hier', 'altoos', 'vorenstaande', 'eigen', 'zodra', 'over', 'enkel', 'lijken', 'iemand', 'van', 'bovendien', 'haar', 'welk', 'ben', 'mag', 'wezen', 'bovenal', 'mr', 'maakten', 'hemzelf', 'sindsdien', 'daarop', 'zulke', 'zover', 'omtrent', 'via', 'hem', 'geweest', 'worden', 'hierboven', 'bovenstaand', 'veel', 'behoudens', 'uitgezonderd', 'überhaupt', 'tiende', 'hadden', 'zo', 'allebei', 'wij', 'twee', 'voordien', 'werder', 'expl', 'ondanks', 'voordat', 'hijzelf', 'hetzelfde', 'bij', 'hierin', 'genoeg', 'derhalve', 'zes', 'vroeg', 'op', 'doet', 'etcetera', 'eerste', 'mits', 'blijken', 'mede', 'thans', 'inzake', 'precies', 'alt', 'tja', 'welke', 'wiens', 'zich', 'vijfde', 'volgens', 'terwijl', 'geen', 'mijnent', 'mochten', 'pas', 'later', 'naar', 'veertig', 'waarop', 'eveneens', 'weg', 'hedden', 'daaruit', 'omstreeks', 'veeleer', 'vijf', 'zelfde', 'hebt', 'evenwel', 'waaruit', 'moeten', 'opzij', 'mijner', 'hiervoor', 'eerdat', 'beetje', 'binnen', 'dikwijls', 'ze', 'u', 'had', 'moesten', 'voordezen', 'gemogen', 'volgend', 'gemoeten', 'net', 'opnieuw', 'onze', 'daarin', 'waaraan', 'omver', 'dhr', 'jezelf', 'me', 'omhoog', 'wier', 'vierde', 'sommige', 'deze', 'met', 'mezelf', 'men', 'ikzelf', 'weer', 'wilden', 'tussen', 'reeds', 'vaak', 'sedert', 'tamelijk', 'een', 'toen', 'minder', 'jijzelf', 'maak', 'namelijk', 'nog', 'cv', 'dit', 'der', 'heeft', 'gehad', 'ofschoon', 'mijzelf', 'dergelijke', 'spoedig', 'in', 'ook', 'tegen', 'onszelf', 'mevr', 'paar', 'zou', 'nam', 'of', 'buiten', 'achterna', 'vooral', 'intussen', 'alsnog', 'óók', 'mw', 'doch', 'uit', 'mij', 'voorheen', 'waaronder', 'daarvan', 'hieruit', 'nadien', 'nader', 'daarmee', 'dusver', 'hiernaast', 'hierna', 'eraan', 'eraf', 'erbij', 'erbovenop', 'erdoor', 'erdoorheen', 'erin', 'ermee', 'ernaast', 'erom', 'eronder', 'eronderdoor', 'erop', 'eropaf', 'eropuit', 'ertegenaan', 'ertoe', 'ertussen', 'ertussenuit', 'eruit', 'ervan', 'ervanaf', 'ervandoor', 'ervantussen', 'hierbij', 'hierdoor', 'hierheen', 'hierin', 'hieronder', 'hieronderdoor', 'hierop', 'hieruit', 'hiervan', 'hiervoor', 'daaraan', 'daarachter', 'daaruit', 'daarbij', 'daarbinnen', 'daarboven', 'daardoor', 'daarheen', 'daarin', 'daarmee', 'daarmede', 'daarna', 'daarnaast', 'daaronder', 'daarop', 'daarover', 'daartoe', 'daartussen', 'daarvandaan', 'daarvan', 'daarvoor', 'waarbij', 'waarvan', 'waaraan', 'waarop', 'waarnaar', 'waarin' 'waaruit', 'waarvoor', 'waarover', 'waartoe', 'waaronder', 'waarheen', 'waaruit', 'waarachter', 'waarom', 'waarmee', 'door_gaan', 'boven_vermen', 'boven_noemen', 'boven_staan', 'uit_zonderen', 'al_tallen']
  merged_df_pos_ner_filtered_5 = merged_df_pos_ner_filtered_4.loc[~merged_df_pos_ner_filtered_4['word'].isin(del_word)]
  merged_df_pos_ner_filtered_6 = merged_df_pos_ner_filtered_5.loc[~merged_df_pos_ner_filtered_5['lemma'].isin(del_word)]

  merged_df_pos_ner_filtered_7 = merged_df_pos_ner_filtered_6[~merged_df_pos_ner_filtered_6['word'].str.contains(r'\.', regex=True)].copy()

  return merged_df_pos_ner_filtered_7

##2.4. Получение дата-фрейма для выявления ключевых слов - get_data_frame_key_words
Функция чистит первичный дата-фрейм, собранный в предыдущей функции, для его дальнейшего использования при подсчете TF-IDF.

Из первичного дата-фрейма удаляются типичные для юридического языка и / или судебных решений слова, такие как "жалоба", "местонахождение", названия месяцев и другие. Эти слова не вычищаются с помощью TF-IDF, потому что присутствуют не в каждом тексте судебного решения, а если присутствуют, то в разных объемах - так, что иногда TF-IDF начинает их выделять в качестве ключевых слов, в то время как по этим словам невозможно составить представление о сути судебного разбирательства.

Слова отобраны частично на основе данных из интернета о типичной юридической терминологии, частично вручную - по итогам первой апробации функции на корпусе текстов судебных решений. Таким образом, в списке юридических стоп-слов могут быть собраны не все слова, которые могли бы туда попасть при отборе слов на основе обработки большого корпуса правовых текстов (такая задача не вошла в задачи проекта).

In [9]:
def get_data_frame_key_words(df_common):
  # см. комментарий про стоп-слова в коде в п. 2.3
  del_words = ['bodemprocedure', 'rechtsmacht', 'rechter', 'wettelijk', 'eiseres', 'kantoorhouden', 'uit_gaan', 'raad_sheer', 'genaamd', 'hoog', 'eindarrest', 'lid', 'maart', 'griffier', 'cashdepot', 'hypotheekviseur', 'raadvrouw', 'verzoekster', 'costa', 'ricaanse', 'rechtsvordering', 'get', 'verzetarrest', 'eiseressen', 'reageren', 'ehg', 'procesverloop', 'whw', 'vermelding', 'hoogte', 'verstekarrest', 'jas', 'hoeven', 'moeder', 'spreken', 'leggen', 'wethouder', 'rzv', 'kilometer', 'kosten', 'zetelen', 'abc', 'vof', 'belanghebbende', 'raad', 'verwerping', 'conclusie', 'vestigingplaats', 'wamrichtlijn', 'betrokkene', 'tijdelijk', 'verzoeksters', 'rechtbank', 'vordering', 'partij', 'governance', 'lwb', 'curator', 'organizatie', 'nummer', 'verweersters', 'ondertekenen', 'zaak', 'heden', 'president', 'caot', 'titel', 'kvz', 'klachtprocedure', 'beslissing', 'weesp', 'eugs', 'bepaling', 'juridisch', 'wvggz', 'klachtbeoordelen', 'doen', 'voorwaardelijk', 'zus', 'verweerder', 'uitspreken', 'aangifte', 'doel', 'verzoeker', '290bedrijfsruimte', 'verweer', 'raadsheer', 'commissaris', 'stul', 'beoordeling', 'incidenteel', 'gezamenlijk', 'aanhef', 'toezegging', 'indienen', 'procureur', 'statuut', 'februari', 'eindvonnis', 'herstelarrest', 'geven', 'verweerschrift', 'verbaal', 'ems', 'caobepaling', 'verschijnen', 'lovers', 'cassatie', 'bns', 'divi', 'rol', 'gerechtshof', 'rit', 'naam', 'eye', 'aanleg', 'eindbeslissing', 'voegen', 'zitiing', 'instellen', 'rechtercommissaris', 'uitgangspunt', 'advocaat', 'arrondissement', 'oktober', 'principaal', 'azm', 'kayooms', 'tussenvonnis', 'vader', 'bericht', 'kubus', 'geding', 'wetboek', 'veste', 'gerecht', 'wet', 'feit', 'mei', 'hof', 'akte', 'verzoekschrift', 'vennootschap', 'pot', 'vennot', 'cassatieberoep', 'bruto', 'dusver', 'achtereenvolgend', 'tan', 'verweerders', 'november', 'instantie', 'eiser', 'tweede', 'wanbeleid', 'eisers', 'categorie', 'tegenbewijs', 'cogsa', 'pkn', 'schriftelijk', 'rechtvaardiging', 'hoedanigheid', 'verz', 'urw', 'ter', 'zijde', 'meter', 'september', 'artikel', 'vestigen', 'kost', 'prejudiciële', 'kantonzaak', 'officier', 'beschikking', 'oordeel', 'gedingstuk', 'bewijs', 'toelichten', 'cao', 'termijn', 'openbaar', 'rvverzoek', 'bws', 'evrm', 'verwijzen', 'wzd', 'ue', 'megaapbbennootschap', 'datum', 'incidentelen', 'verlenen', 'uu', 'voorhand', 'verweerster', 'wijziging', 'eindbeschikking', 'uitkomst', 'voetnoot', 'sagel', 'college', 'verzoekers', 'recht', 'herstelvonnis', 'augustus', 'verloop', 'fase', 'cnb', 'rechthebbende', 'verstek', 'juni', 'catchers', 'vandaag', 'wam', 'tussentijds', 'wan', 'kort', 'maand', 'justitie', 'mhr', 'extra', 'kantonrechter', 'december', 'belang', 'voorzitter', 'oud', 'midden', 'vraag', 'nai', 'verzoek', 'regel', 'april', 'rechterlijk', 'wetgeving', 'maken', 'zaaknummer', 'klacht', 'gemeente', 'middel', 'moving', 'brief', 'aua', 'januari', 'rechtvraag', 'vonnis', 'rele', 'uitstrekken', 'miljard', 'juli', 'volgen', 'man', 'advocaat-generaal', 'broer', 'belanghebben', 'deskundig', 'antwoord', 'raadsman', 'feitelijk', 'heusden', 'incident', 'adres', 'vroeg', 'staatssecretaris', 'op', 'ondernemingskamer', 'beroep', 'richtlijn', 'maatschap', 'dragen', 'principale', 'maatregel', 'jaar', 'arbeidsomstandighedenwetenwetenwetenwetenweten', 'karakter', 'de instelling', 'vicepresident', 'dws', 'uitspraak', 'mail', 'vierkant', 'leiden', 'zien', 'stil', 'sxm', 'vrouw', 'proces', 'dag', 'wonen', 'komen', 'arrest', 'procesverloop', 'arrondissement', 'organisatie', 'nodig', 'wettelijk', 'respectief', 'stellen', 'somalisch', 'stuk', 'aantal', 'cassatieprocedure', 'luiden', 'plaatsvinden', 'jongmeerderderjarig', 'wetboek', 'wet', 'vallen', 'rechtsmacht', 'costa', 'ricaans', 'laat', 's-hertogenbosch', 'kantoorhoudend', 'vorm', 'verzoeken', 'woonadre', 'aruba', 'curaçao', 'sint Maarten', 'sint', 'maarten', 'megaapbbennootschap', 'meidooornaag', 'lisse', 'schadetaatprocedure', 'jongmeerderderjarig', 'kantoor_houd', 'eind_arrest', 'hypotheek_viseur', 'proces_verloop', 'kilo_meter', 'belang_hebben', 'vestiging_plaats', 'klacht_procedure', 'klacht_beoordeel', 'uit_spreken', '290bedrijf_ruimte', 'in_dienen', 'toe_voegen', 'in_stellen', 'wet_boek', 'cassatie_beroep', 'achtereen_volgen', 'wan_beleid', 'kanton_zaak', 'toe_lichten', 'megaap_bennootschap', 'eind_beschikking', 'herstel_vonnis', 'zaak_nummer', 'uit_strekken', 'richt_lijn', 'arbeid_omstandighedenwetenwetenwetenwet', 'proces_verloop', 'cassatie_procedure', 'plaats_vinden', 'wet_boek', 'megaap_bennootschap', 'meid_ooornaag', 'schadetaat_procedure']
  df_for_key_words = df_common.loc[~df_common['word'].isin(del_words)]
  df_for_key_words_2 = df_for_key_words.loc[~df_for_key_words['lemma'].isin(del_words)]

  return df_for_key_words_2

##2.5. Подсчет TTR - get_ttr
Функция считает TTR (type-token ratio) для оценки лексического разнообразия текста судебного решения.

Поскольку тексты судебных решений в корпусе разного размера, используется стандартизированный TTR с окном в 200 лемм. При этом для текстов, в которых меньше 200 лемм, TTR хоть и считается, но в дальнейшем не используется в чат-боте, чтобы не исказить данные. Кроме того, для маленьких текстов судебных решений TTR непоказателен. Небольшие тексты судебных решений типовые, в них обычно содержится лишь отсылка на решения нижестоящих судов и указание на то, что Верховный суд поддерживает эти решения.


Также учтено, что окно в 200 лемм по-разному укладывается в тексты судебных решений: где-то последнее окно близко к 200, а где-то намного меньше, что может приводить к безосновательному завышению TTR. Чтобы смягчить этот эффект, последнее окно, в зависимости от его размера, либо считается отдельно (если оно ближе к 200, чем к 0), либо присоединяется к предыдущему (если оно ближе к 0, чем к 200).

In [10]:
def get_ttr(df_1):
  df = df_1.copy()
  df['lemma'] = df['lemma'].apply(lambda x: re.sub('_', '', x))
  if df.shape[0] <= 200:
    num_types = df['lemma'].nunique()
    num_tokens = df.shape[0]
    ttr = round((num_types / num_tokens)*100, 3)

  elif df.shape[0] > 200:
    lemma_list = df['lemma'].tolist()
    number_of_lemmas = len(lemma_list)

    ttr_200_all = []
    n = 0
    k = 200
    count = 0

    while count <= (number_of_lemmas/200):
      if k <= number_of_lemmas:
        types = len(set(lemma_list[n:k]))
        ttr_200 = round((types / 200)*100, 3)
        ttr_200_all.append(ttr_200)
        n = n + 201
        k = k + 201
      elif (k > number_of_lemmas) and ((number_of_lemmas/200 % 1) >= 0.5):
        types = len(set(lemma_list[n:]))
        ttr_200 = round((types / 200)*100, 3)
        ttr_200_all.append(ttr_200)
      elif (k > number_of_lemmas) and ((number_of_lemmas/200 % 1) < 0.5):
        del ttr_200_all[-1]
        n = n - 201
        types = len(set(lemma_list[n:]))
        ttr_200 = round((types / 200)*100, 3)
        ttr_200_all.append(ttr_200)
      count = count + 1
    ttr = round(sum(ttr_200_all)/len(ttr_200_all), 3)

  return ttr

## 2.6. Замена неверно лемматизированных в Stanza сложных слов - complex_words

Функция подготавливает слова (собирает их из полученного ранее дата-фрейма), на основе которых рассчитывается TF-IDF.

Особенность функции заключается в том, что вместо того, чтобы просто собрать все леммы из дата-фрейма, она сначала делит слова на сложные (состоящие из нескольких корней прилагательные и существительные) и простые и для простых собирает леммы, а для сложных - их словоформы.

Такое разделение необходимо, чтобы не допускать орфографических ошибок при выдаче пользователю ключевых слов, поскольку лемматизации сложных слов, состоящих из нескольких корней (существительных и прилагательных), Stanza иногда удаляет / добавляет буквы в середине слова. Из-за этого при выдаче ключевых слов сложные слова отображаются с ошибкой. Например: в качестве леммы 'mededeling**s**plicht' Stanza выдает 'mededeling_plicht', видимо, удаляя 's' как соединительную гласную; для 'jongmeerderjarig' выдает 'jongmeer**der_der**jarig'.

Чтобы такие ошибки в ключевых словах нивелировать, было принято решение для сложных прилагательных и существительных собирать не их леммы, а их словоформы, то есть брать их в том виде, в котором они встретились в тексте.

Сбор словоформ вместо лемм мог бы оказать негативное влияние на состав ключевых слов, потому что для корректного подсчета ключевых слов необходимо собирать слова в начальной форме, чтобы не искажать подсчеты за счет различий в словоформах. Однако в данном случае, как представляется, результат в целом оказывается одним и тем же, потому что лемма у слова некорректная и всё равно не совпадает с действительной леммой слова. Кроме того, у существительных и прилагательных в нидерландском языке не так много форм (в отличие, например, от глаголов), только единственное и множественное число. Поскольку программа предназначена для обучающихся и преподавателей, в итоге выбор был сделан в пользу корректного (без орфографических ошибок) отображения ключевых слов, несмотря на возможные риски, связанные с негативным влиянием сбора словоформ вместо лемм на состав (набор) ключевых слов.

Функция не используется при подсчете TTR, поскольку, как представляется, для TTR непринципиально, корректная ли лемма, важно только то, уникальная она или нет.

In [11]:
def complex_words(data_frame):
  look_for = ['NOUN', 'ADJ']

  filter_line = data_frame.loc[(data_frame['upos'].isin(look_for)) & ((data_frame['lemma'].str.contains('_'))|(data_frame['word'].str.contains('eiseres')))]
  lemmas_list_complex = filter_line['lemma'].tolist()
  words_list_complex = filter_line['word'].tolist()

  filter_not_line = data_frame.loc[~data_frame['lemma'].isin(lemmas_list_complex)].copy()
  filter_not_line['lemma'] = filter_not_line['lemma'].apply(lambda x: re.sub('_', '', x))
  lemmas_list_not_complex = filter_not_line['lemma'].tolist()

  all_words_and_lemmas = words_list_complex + lemmas_list_not_complex

  return all_words_and_lemmas

#3. Вспомогательные подсчеты

##3.1. Поиск слов в квадратных скобках
Найти и проанализировать слова в квадратных скобках понадобилось, чтобы затем их корректно почистить внутри функции final_clean_the_text.

Для этого все судебные решения были сначала для удобства собраны в один файл, а затем в этом файле с помощью регулярных выражений были отобраны все слова в квадратных скобках. При этом чтобы сосредоточиться в рамках функции final_clean_the_text только на более частотных словах, они были подсчитаны в Counter и визуализированы в дата-фрейме по убыванию. В данном случае было решено использовать абсолютные частоты в связи с тем, что они использовались лишь в качестве общего ориентира при чистке слов, которые затем отбирались вручную.

In [ ]:
file = open('/content/Корпус одним файлом.txt', 'a', encoding='utf-8')
for i in os.listdir():
  if i.startswith('ECLI'):
    with open(f'/content/{i}', 'r', encoding='utf-8') as file_by_one:
      decision = file_by_one.read()
      decision_clean = initial_clean_the_text(decision)
      file.write(decision_clean)
file.close()

In [ ]:
with open('/content/Корпус одним файлом.txt', encoding = 'utf-8') as c:
  all_decisions = c.read()
words_in_brackets = re.findall(r'\[[\w\s/]+\]', all_decisions)
words_in_brackets_2 = [i.lower() for i in words_in_brackets]

data_frame_wib = pd.DataFrame.from_dict(Counter(words_in_brackets_2), orient='index', columns=['count'])
data_frame_wib_sorted = data_frame_wib['count'].sort_values(ascending=False)
data_frame_wib_sorted.to_excel('counter_brackets_all.xlsx')
data_frame_wib_sorted

##3.2. Составление списка сокращенных названий нормативных актов
В судебных решениях Верховного суда Нидерландов названия законов и иных нормативных актов часто сокращаются (например, BW - Burgerlijk Wetboek). Иногда сокращение конкретного акта в судебном решении оказывается настолько частотным, что попадает в список ключевых слов - но такое ключевое слово непоказательно для пользователя, потому что сокращение нужно ещё расшифровать, чтобы понять, чему посвящен нормативный акт, а ключевое слово должно сразу давать представление о содержании судебного решения.

В связи с этим было принято решение сокращения названий актов в корпусе заменить на слово-заглушку (wetboek, кодекс). Для этого был составлен список наиболее частотных сокращений, которые встречаются в текстах судебных решений. Сокращения были взяты с сайта нидерландской Википедии.

In [ ]:
!pip install requests
!pip install beautifulsoup4
!pip install fake_useragent

In [ ]:
import requests
from fake_useragent import UserAgent

In [ ]:
page_link = 'https://nl.wikipedia.org/wiki/Lijst_van_afkortingen_voor_wetten_en_regelingen'
response = requests.get(page_link, headers={'User-Agent': UserAgent().chrome})

In [ ]:
afkorting = pd.read_html(response.text)

In [ ]:
list_of_lists = []
for i in range(len(afkorting)):
  df_af = afkorting[i].loc[~afkorting[i]['Betekenis(sen)'].str.contains('BE', na=False)]
  try:
    list_af = df_af['Afkorting'].tolist()
  except:
    list_af = df_af['Afkortingen'].tolist()

  list_of_lists.append(list_af)

list_final = []
for i in list_of_lists:
  for af in i:
    try:
      list_final.append(af)
    except:
      pass

#4. Создание csv-файла для базы данных чат-бота

##4.1. Формирование дата-фрейма (без ключевых слов)

В данной части кода формируется базовый дата-фрейм, к которому в последующем будет присоединена колонка с ключевыми словами.

В дата-фрейме хранится следующая информация:  
- название файла с судебным решением, в котором содержится его ECLI-код (код в базе данных судебных решений Нидерландов на сайте Rechtspraak.nl);
- порядковый номер судебного решения в дата-фрейме;
- длина судебного решения, подсчитанная через количество лемм;  
- TTR судебного решения, полученный в результате применения функции get_ttr;  
- текст судебного решения, почищенный с помощью функции initial_clean_the_text.

Также одновременно с формированием дата-фрейма собирается необходимая информация для дальнейшего подсчета TF-IDF - в list_of_texts_lemmatized собираются списки лемм текстов судебных решений.

In [18]:
dic = {}
list_of_texts_lemmatized = []

count = 0

for i in os.listdir():
  if i.startswith('ECLI'):
    with open(f'/content/{i}', 'r', encoding='utf-8') as file_by_one:
      decision = file_by_one.read()

    data_frame_decision = get_data_frame_common(final_clean_the_text(initial_clean_the_text(decision)))

    length_decision = data_frame_decision.shape[0]

    ttr_decision = get_ttr(data_frame_decision)

    data_frame_decision_key = get_data_frame_key_words(data_frame_decision)

    lemmas_list = complex_words(data_frame_decision_key)

    lemmas_list_clean = []

    for lemma in lemmas_list:
      lemma_clean = re.sub(r'(?<=[^A-Za-z])-', '', lemma)
      lemma_clean = re.sub(r'(?<=^)-', '', lemma_clean)
      lemmas_list_clean.append(lemma_clean)

    list_of_texts_lemmatized.append(lemmas_list_clean)

    dic[i] = [count, length_decision, ttr_decision, initial_clean_the_text(decision)]

    count = count + 1
    print(f'Added! {count}')

Added! 1
Added! 2
Added! 3
Added! 4
Added! 5


In [19]:
df_corpus = pd.DataFrame.from_dict(dic).T.rename(columns={0:'id_text', 1:'length', 2:'ttr', 3:'text'}).reset_index(names=['ECLI_code'])
df_corpus.to_csv('df_corpus_step_1.csv')
df_corpus.head()

,ECLI_code,id_text,length,ttr,text
0,ECLINLHR202555.txt,0,94,71.277,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...
1,ECLINLHR202586.txt,1,447,46.5,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...
2,ECLINLHR202556.txt,2,567,58.833,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...
3,ECLINLHR202554.txt,3,103,73.786,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...
4,ECLINLHR202585.txt,4,106,75.472,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...


##4.2. Добавление ключевых слов в дата-фрейм
Список лемм текстов направляется в TfidfVectorizer для подсчета ключевых слов. При этом собственный встроенный токенизатор TfidfVectorizer отключается с помощью наивной функции. Такой подход был избран после того, как в выдаче TfidfVectorizer были обнаружены чрезмерно (для целей проекта) токенизированные слова. Например, TfidfVectorizer разбивал слова, написанные через дефис, на несколько слов, такие как экс-супруг(ex-echtgenoot), не-участвующий-в-сделке (niet-handelende).

Для каждого текста на основе результатов подсчета TF-IDF отбираются 10 ключевых слов. Они сохраняются в список вместе с порядковым номером судебного решения и далее присоединяются к основному дата-фрейму по данному порядковому номеру.

После этого формируется последняя колонка дата-фрейма - в нем содержатся вместе ключевые слова и тексты судебных решений. Отсюда чат-бот будет получать текст судебного решения с ключевыми словами, который нужно будет направить пользователю.

In [20]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x)
tfidf_matrix = tfidf_vectorizer.fit_transform(list_of_texts_lemmatized)
feature_names = tfidf_vectorizer.get_feature_names_out()

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [21]:
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
df_tfidf.to_excel('tfidf corpus full.xlsx')
df_tfidf.head()

,aanduiden,aanhalen,aanleiding,aansluiten,aansluiting,aanspraak,aanvang,aanvoeren,aanwezig,aanzien,...,wijzigen,zeggen,zin,zodanig,zogenoemd,zorg,zorgmachtiging,zoverre,zozeer,‘ledenovereenkomst
0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.025945,0.125594,0.05189,0.000000,0.000000,0.000000,0.000000,0.05189,...,0.000000,0.000000,0.020932,0.000000,0.000000,0.05189,0.674572,0.000000,0.000000,0.000000
2,0.023196,0.023196,0.000000,0.018715,0.00000,0.046392,0.023196,0.069589,0.023196,0.00000,...,0.023196,0.069589,0.074858,0.023196,0.023196,0.00000,0.000000,0.023196,0.023196,0.023196
3,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000


In [22]:
list_for_df_to_join = []
for i in range(df_tfidf.shape[0]):
  doc_scores = df_tfidf.iloc[i]
  sorted_scores_doc = doc_scores.sort_values(ascending=False)
  df_top = sorted_scores_doc.head(10)
  top_list_words = df_top.index.tolist()
  list_for_df_to_join.append([i, ', '.join(top_list_words)])
list_for_df_to_join[0]

[0,
 'geheim, beoordelen, eenheid, vernietiging, ontwikkeling, motiveren, verwerpen, richten, rente, regeling']

In [23]:
df_to_join = pd.DataFrame(list_for_df_to_join).rename(columns={0:'id_text', 1: 'key_words'})
df_to_join['key_words'] = df_to_join['key_words'].apply(lambda x: re.sub(r'[a-z]\.[a-z]\.[a-z]\.[a-z]?[\.]?,', '', x))
df_to_join.to_excel('df_to_join.xlsx')
df_to_join.head()

,id_text,key_words
0,0,"geheim, beoordelen, eenheid, vernietiging, ont..."
1,1,"zorgmachtiging, duur, maximaal, uiterlijk, mac..."
2,2,"overeenkomst, grondslag, statutair, vergoeding..."
3,3,"kantoorhouden, nihil, faillissement, veroordel..."
4,4,"politie, rente, verschot, salaris, vermeerdere..."


In [24]:
df_corpus_full = pd.merge(df_corpus, df_to_join, on=['id_text'], how='outer')
df_corpus_full['key_words'] = df_corpus_full['key_words'].apply(lambda x: re.sub(r'_', '', x))
df_corpus_full['text_key_words'] = df_corpus_full['key_words'] + '\n' + df_corpus_full['text']
df_corpus_full.to_csv("df_corpus_full_key_ttr.csv", index=False)
df_corpus_full.head()

,ECLI_code,id_text,length,ttr,text,key_words,text_key_words
0,ECLINLHR202555.txt,0,94,71.277,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...,"geheim, beoordelen, eenheid, vernietiging, ont...","geheim, beoordelen, eenheid, vernietiging, ont..."
1,ECLINLHR202586.txt,1,447,46.5,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...,"zorgmachtiging, duur, maximaal, uiterlijk, mac...","zorgmachtiging, duur, maximaal, uiterlijk, mac..."
2,ECLINLHR202556.txt,2,567,58.833,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...,"overeenkomst, grondslag, statutair, vergoeding...","overeenkomst, grondslag, statutair, vergoeding..."
3,ECLINLHR202554.txt,3,103,73.786,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...,"kantoorhouden, nihil, faillissement, veroordel...","kantoorhouden, nihil, faillissement, veroordel..."
4,ECLINLHR202585.txt,4,106,75.472,Uitspraak\nHOGE RAAD DER NEDERLANDEN\n\nCIVIEL...,"politie, rente, verschot, salaris, vermeerdere...","politie, rente, verschot, salaris, vermeerdere..."


## 5. Пример работы функций

In [12]:
with open('/content/ECLINLHR2025186.txt', 'r', encoding = 'utf-8') as file:
  example = file.read()
print(example[:153])
print('\n----------\n')
print(example[510:699])

Uitspraak
HOGE RAAD DER NEDERLANDEN

CIVIELE KAMER

Nummer 23/04442

Datum 7 februari 2025

ARREST

In de zaak van

ASTARTE B.V.,

gevestigd te Den Haag,

----------


Voor het verloop van het geding in feitelijke instanties verwijst de Hoge Raad naar:

a. de vonnissen in de zaak C/10/447594 / HA ZA 14-336 van de rechtbank Rotterdam van 15 oktober 2014, 


In [13]:
example_clean = final_clean_the_text(initial_clean_the_text(example))
print(example_clean[:140])
print('\n----------\n')
print(example_clean[395:565])

Uitspraak Hoge Raad Der Nederlanden Civiele Kamer nummer 23/04442 datum 7 februari 2025 Arrest In de zaak van Gert , gevestigd te Den Haag, 

----------

Voor het verloop van het geding in feitelijke instanties verwijst de Hoge Raad naar: a. de vonnissen in de zaak nummer 123 van de rechtbank Rotterdam van 15 oktober 2014,


In [14]:
df_common_example = get_data_frame_common(example_clean)

In [16]:
df_common_example

,sentence_id,word_id,word,lemma,upos,xpos,head,deprel,ner
0,0,1,uitspraak,uitspraak,NOUN,N|soort|ev|basis|zijd|stan,0,root,O
7,0,8,nummer,nummer,NOUN,N|soort|ev|basis|onz|stan,1,appos,O
9,0,10,datum,datum,NOUN,N|soort|ev|basis|zijd|stan,11,case,O
11,0,12,februari,februari,PROPN,N|eigen|ev|basis|zijd|stan,11,flat,O
13,0,14,arrest,arrest,NOUN,N|soort|ev|basis|onz|stan,11,appos,O
...,...,...,...,...,...,...,...,...,...
3586,105,7,vicepresident,vicepresident,NOUN,N|soort|ev|basis|zijd|stan,4,obl:agent,O
3590,105,11,voorzitter,voorzitter,NOUN,N|soort|ev|basis|zijd|stan,4,xcomp,O
3593,105,14,raadsheren,raad_sheer,NOUN,N|soort|mv|basis,11,conj,O
3607,105,28,openbaar,openbaar,ADJ,ADJ|nom|basis|zonder|zonder-n,29,obl,O


In [15]:
df_common_example.head()
# df_common_example.to_excel('example_186_df_common.xlsx')

,sentence_id,word_id,word,lemma,upos,xpos,head,deprel,ner
0,0,1,uitspraak,uitspraak,NOUN,N|soort|ev|basis|zijd|stan,0,root,O
7,0,8,nummer,nummer,NOUN,N|soort|ev|basis|onz|stan,1,appos,O
9,0,10,datum,datum,NOUN,N|soort|ev|basis|zijd|stan,11,case,O
11,0,12,februari,februari,PROPN,N|eigen|ev|basis|zijd|stan,11,flat,O
13,0,14,arrest,arrest,NOUN,N|soort|ev|basis|onz|stan,11,appos,O


In [ ]:
#что происходит внутри функции complex_words
look_for = ['NOUN', 'ADJ']

filter_line = df_common_example.loc[(df_common_example['upos'].isin(look_for)) & ((df_common_example['lemma'].str.contains('_'))|(df_common_example['word'].str.contains('eiseres')))]
lemmas_list_complex = filter_line['lemma'].tolist()
words_list_complex = filter_line['word'].tolist()
print(lemmas_list_complex[20:27]) #для сравнения - при лемматизации в Stanza иногда теряются буквы / иногда дописываются лишние буквы в середине
print(words_list_complex[20:27])

filter_not_line = df_common_example.loc[~df_common_example['lemma'].isin(lemmas_list_complex)].copy()
filter_not_line['lemma'] = filter_not_line['lemma'].apply(lambda x: re.sub('_', '', x))
lemmas_list_not_complex = filter_not_line['lemma'].tolist()
print(lemmas_list_not_complex[:10])


In [17]:
df_key_example = get_data_frame_key_words(df_common_example)
# df_key_example.to_excel('example_186_df_key.xlsx')
df_key_example.head()

,sentence_id,word_id,word,lemma,upos,xpos,head,deprel,ner
171,11,7,gevoerd,voeren,VERB,WW|vd|vrij|zonder,0,root,O
197,13,9,vernietiging,vernietiging,NOUN,N|soort|ev|basis|zijd|stan,7,obl:arg,O
200,13,12,bestreden,bestreden,VERB,WW|vd|prenom|zonder,13,amod,O
204,13,16,verwijzing,verwijzing,NOUN,N|soort|ev|basis|zijd|stan,9,conj,O
206,13,18,terugwijzing,terugwijzing,NOUN,N|soort|ev|basis|zijd|stan,16,conj,O


In [ ]:
TTR = get_ttr(df_common_example)
print(df_common_example.shape[0])
print(TTR)

In [ ]:
#что происходит внутри функции TTR
df_common_example['lemma'] = df_common_example['lemma'].apply(lambda x: re.sub('_', '', x))
df = df_common_example
if df.shape[0] <= 200:
  num_types = df['lemma'].nunique()
  num_tokens = df.shape[0]
  ttr = round((num_types / num_tokens)*100, 3)
  print(f'Лемм меньше 200! TTR = {ttr}')

elif df.shape[0] > 200:
  lemma_list = df['lemma'].tolist()
  number_of_lemmas = len(lemma_list)

  ttr_200_all = []
  n = 0
  k = 200
  count = 0

  while count <= (number_of_lemmas/200):
    if k <= number_of_lemmas:
      types = len(set(lemma_list[n:k]))
      ttr_200 = round((types / 200)*100, 3)
      ttr_200_all.append(ttr_200)
      n = n + 201
      k = k + 201
    elif (k > number_of_lemmas) and ((number_of_lemmas/200 % 1) >= 0.5):
      types = len(set(lemma_list[n:]))
      ttr_200 = round((types / 200)*100, 3)
      ttr_200_all.append(ttr_200)
    elif (k > number_of_lemmas) and ((number_of_lemmas/200 % 1) < 0.5):
      del ttr_200_all[-1]
      n = n - 201
      types = len(set(lemma_list[n:]))
      ttr_200 = round((types / 200)*100, 3)
      ttr_200_all.append(ttr_200)
    count = count + 1
  ttr = round(sum(ttr_200_all)/len(ttr_200_all), 3)
  print(f'Лемм больше 200! TTR = {ttr}, TTR отдельных окон = {ttr_200_all}')